# `.eospar` analysis

EOS `.eospar` files (the contents of `materialset/` inside an `.openjz` build archive) hold the laser/process parameters for an EOSPRINT job. Recent versions of EOSPRINT encrypt the body of the file, so we cannot read the actual parameter values without EOS's keys.

What we *can* do without decrypting:

1. **Parse the unencrypted header and the trailing offset table** to count the laser/build configuration records the file claims to contain.
2. Given **two or more `.eospar` files from the same printer**, run a few cheap forensics tests on the ciphertexts and headers to decide whether they appear to share an encryption key.

This notebook implements both.

In [1]:
from __future__ import annotations

import collections
import math
import struct
from dataclasses import dataclass, field
from pathlib import Path
from typing import Iterable

# Default sample file shipped with this repo.
DEMO_EOSPAR = Path("CMU02_Adjusted_02-19-24_extracted") / "materialset" / "Ti64_030_PerformanceM291_110.eospar"

## 1. The file layout

Empirically (see `eos_file_parsing.ipynb` for the byte-level inspection that motivated this), an `.eospar` looks like:

```
[ tiny TLV header                                            ]
  - a sequence of records, each tagged "LC" (laser config) or "BC" (build config),
    preceded by a 1-byte count/index byte and followed by 0x00
  - then a couple of small uint32-looking fields (probably section sizes)
[ encrypted (and presumably pre-compressed) payload, ~8.000 bits/byte ]
[ trailing offset table                                      ]
  - a packed array of little-endian uint32 values that index into the decrypted payload
```

The header and the trailer are plaintext. The middle is opaque ciphertext. `read_eospar_header()` below parses the two plaintext regions and returns a structured summary.

In [2]:
@dataclass
class EosparHeader:
    path: Path
    file_size: int
    header_bytes: bytes              # the plaintext bytes before the encrypted body
    body_offset: int                 # where the encrypted body starts
    body_size: int                   # length of the encrypted body
    lc_records: list                 # list of (count_byte, raw_record_bytes) for "LC" tags
    bc_records: list                 # same for "BC" tags
    trailer_uint32s: list            # parsed uint32-LE values from the file trailer
    trailer_bytes: bytes             # raw trailer bytes for inspection

    @property
    def n_laser_configs(self) -> int:
        return len(self.lc_records)

    @property
    def n_build_configs(self) -> int:
        return len(self.bc_records)

    def summary(self) -> str:
        return (
            f"{self.path.name}\n"
            f"  file size       : {self.file_size:,} bytes\n"
            f"  header (plain)  : {len(self.header_bytes)} bytes\n"
            f"  encrypted body  : {self.body_size:,} bytes (offset {self.body_offset})\n"
            f"  trailer (plain) : {len(self.trailer_bytes)} bytes\n"
            f"  laser configs   : {self.n_laser_configs}  (LC tags)\n"
            f"  build configs   : {self.n_build_configs}  (BC tags)\n"
            f"  trailer uint32s : {[hex(v) for v in self.trailer_uint32s]}"
        )


def _scan_tlv_header(data: bytes, scan_limit: int = 256):
    """Walk the plaintext TLV header.

    Each record observed in the wild looks like:  <count_byte> 'L' 'C' 0x00
    or                                            <count_byte> 'B' 'C' 0x00
    with one observed variant where an extra byte appears between the count and
    the tag (e.g. 0x03 0x83 'L' 'C' 0x00). We accept either form.

    Returns: (lc_records, bc_records, header_end_offset)
    """
    lc, bc = [], []
    i = 0
    end = min(scan_limit, len(data))
    while i < end - 3:
        # Try the 4-byte form: <count> <tag2> 0x00
        tag = data[i + 1 : i + 3]
        if tag in (b"LC", b"BC") and data[i + 3] == 0:
            count = data[i]
            rec = bytes(data[i : i + 4])
            (lc if tag == b"LC" else bc).append((count, rec))
            i += 4
            continue
        # Try the 5-byte form: <count> <extra> <tag2> 0x00
        if i + 4 < end:
            tag5 = data[i + 2 : i + 4]
            if tag5 in (b"LC", b"BC") and data[i + 4] == 0:
                count = data[i]
                rec = bytes(data[i : i + 5])
                (lc if tag5 == b"LC" else bc).append((count, rec))
                i += 5
                continue
        # No tag at this position — header has ended.
        break
    return lc, bc, i


def _parse_trailer_uint32s(data: bytes, max_trailer: int = 64) -> tuple[list[int], bytes]:
    """Read the last `max_trailer` bytes as little-endian uint32 values.

    The trailer length is not self-describing in the format we observe, so we
    just expose a fixed window. The caller can decide how many entries are
    meaningful.
    """
    n = min(max_trailer, len(data))
    n -= n % 4  # round down to a multiple of 4 so struct can unpack cleanly
    tail = data[-n:] if n else b""
    values = list(struct.unpack("<" + "I" * (n // 4), tail))
    return values, tail


def read_eospar_header(path: str | Path, *, scan_limit: int = 256, trailer_bytes: int = 64) -> EosparHeader:
    """Parse the plaintext TLV header and trailing offset table of an .eospar file.

    We do **not** decrypt the body. The returned object reports how many laser
    configuration (LC) and build configuration (BC) records the header advertises,
    where the encrypted body sits, and the raw uint32 values in the trailer.
    """
    path = Path(path)
    data = path.read_bytes()
    lc, bc, header_end = _scan_tlv_header(data, scan_limit=scan_limit)
    trailer_vals, trailer = _parse_trailer_uint32s(data, max_trailer=trailer_bytes)
    return EosparHeader(
        path=path,
        file_size=len(data),
        header_bytes=bytes(data[:header_end]),
        body_offset=header_end,
        body_size=len(data) - header_end - len(trailer),
        lc_records=lc,
        bc_records=bc,
        trailer_uint32s=trailer_vals,
        trailer_bytes=trailer,
    )

### Demo on the bundled file

Run the parser on the sample shipped with the repo. The header should report **5 LC + 1 BC** records, and the trailer should expose a small ascending list of uint32 offsets.

In [3]:
demo = read_eospar_header(DEMO_EOSPAR)
print(demo.summary())
print()
print("raw header bytes:")
print(" ", demo.header_bytes.hex(" "))

Ti64_030_PerformanceM291_110.eospar
  file size       : 661,773 bytes
  header (plain)  : 25 bytes
  encrypted body  : 661,684 bytes (offset 25)
  trailer (plain) : 64 bytes
  laser configs   : 5  (LC tags)
  build configs   : 1  (BC tags)
  trailer uint32s : ['0x98283e16', '0x989a214', '0x9bcbfed9', '0x2c623b06', '0x350b78e4', '0x0', '0x18780', '0x31050', '0x49950', '0x62230', '0x7aa30', '0x93300', '0x9c360', '0xa18c0', '0x9', '0xbdb65']

raw header bytes:
  0c 4c 43 00 0b 4c 43 00 03 83 4c 43 00 05 4c 43 00 11 4c 43 00 02 42 43 00


## 2. Same-key forensics across multiple `.eospar` files

We can't recover the key, but if we have several `.eospar` files from the same printer (and ideally the same EOSPRINT install / license) we can usually decide whether they were encrypted under the **same** key by running four cheap tests. None of them require decrypting the body.

| # | Test | What a hit means | What a miss means |
|---|------|------------------|-------------------|
| 1 | **Header diff** — align the plaintext headers byte-by-byte | A constant ~8–32-byte "random-looking" run is a strong key-id / KCV candidate; same value across files ⇒ same key (or same KEK). | Constant key-id-shaped field with **different** values ⇒ different keys. |
| 2 | **Cross-file 16-byte block collision** | Even one repeated 16-byte ciphertext block across two files is statistically impossible by chance (≈10⁻²⁹); proves ECB **and** shared key. | No collision ⇒ either not ECB, or compressed-then-ECB, or different keys. Inconclusive. |
| 3 | **XOR of two bodies, entropy test** | Entropy noticeably below 8.0 bits/byte (or readable cribs visible in the XOR) ⇒ stream cipher with reused key **and** nonce. | Entropy ~8.0 ⇒ rules out (same key + same nonce) only. Doesn't rule out same key with different nonces. |
| 4 | **Layout fingerprint** | Same record counts and consistent body-size offsets across files ⇒ same encryption pipeline; necessary, not sufficient, for shared key. | Different layouts ⇒ probably different format versions; comparing keys is moot. |

The combined `shared_key_verdict()` at the bottom rolls these into a single judgement.

### Loading a directory of files

In [4]:
def load_eospar_files(paths: Iterable[str | Path]) -> list[tuple[EosparHeader, bytes]]:
    """Read each .eospar file and return a list of (header, body_bytes) tuples.

    `body_bytes` is the encrypted payload *only* (header and trailer stripped),
    which is what tests 2 and 3 operate on.
    """
    out = []
    for p in paths:
        hdr = read_eospar_header(p)
        raw = Path(p).read_bytes()
        body = raw[hdr.body_offset : hdr.body_offset + hdr.body_size]
        out.append((hdr, body))
    return out

### Test 1 — Header diff

Align the plaintext headers up to their shortest common length and classify each byte position:

- **constant** across all files → format/version/machine field, or a **key id / KCV** if it looks random.
- **varying** across files → either a per-file IV/nonce/salt (random-looking) or a record count/size (structured).

We then highlight the longest constant runs of ≥8 bytes — those are the prime key-id candidates.

In [5]:
def test_header_diff(files: list[tuple[EosparHeader, bytes]], min_run: int = 8):
    headers = [h.header_bytes for h, _ in files]
    if len(headers) < 2:
        print("need at least 2 files for a header diff")
        return None

    L = min(len(h) for h in headers)
    constant_mask = [all(h[i] == headers[0][i] for h in headers) for i in range(L)]

    # Find runs of constant positions.
    runs, run_start = [], None
    for i, c in enumerate(constant_mask + [False]):
        if c and run_start is None:
            run_start = i
        elif not c and run_start is not None:
            runs.append((run_start, i - run_start))
            run_start = None

    n_const = sum(constant_mask)
    print(f"compared {len(headers)} headers over {L} byte positions")
    print(f"  constant positions : {n_const} / {L}")
    print(f"  varying positions  : {L - n_const} / {L}")

    long_runs = [r for r in runs if r[1] >= min_run]
    if long_runs:
        print(f"\nlong constant runs (>= {min_run} bytes) — these are the key-id / KCV / version candidates:")
        for off, length in long_runs:
            value = headers[0][off : off + length]
            print(f"  offset 0x{off:04x}  length {length:3d}  value {value.hex(' ')}")
    else:
        print(f"\nno constant runs of >= {min_run} bytes")

    # Also print which positions vary, with each file's value, for the curious.
    varying = [i for i, c in enumerate(constant_mask) if not c]
    if varying:
        print(f"\nvarying byte positions: {varying[:32]}{' ...' if len(varying) > 32 else ''}")
        for h, name in zip(headers, [f.path.name for f, _ in files]):
            shown = bytes(h[i] for i in varying[:32]).hex(' ')
            print(f"  {name:50s}  {shown}")

    return {
        "compared_length": L,
        "n_constant": n_const,
        "long_constant_runs": long_runs,
        "varying_positions": varying,
    }

### Test 2 — Cross-file 16-byte block collision

Slice every encrypted body into 16-byte blocks and look for any block value that appears in two or more *different* files. Among ~10⁵ random 128-bit blocks the probability of even one collision is ~10⁻²⁹, so any cross-file hit is conclusive evidence of (a) ECB-mode encryption and (b) a shared key.

If the plaintext is compressed before being ECB-encrypted then this test won't find anything either way — a miss is therefore inconclusive, but a hit is a smoking gun.

In [6]:
def test_block_collisions(files: list[tuple[EosparHeader, bytes]], block: int = 16):
    # Map block -> set of file indices it appears in.
    seen: dict[bytes, set[int]] = collections.defaultdict(set)
    for fi, (_, body) in enumerate(files):
        for i in range(0, len(body) - block + 1, block):
            seen[body[i : i + block]].add(fi)

    cross = {b: idxs for b, idxs in seen.items() if len(idxs) > 1}
    total_blocks = sum((len(b) // block) for _, b in files)
    print(f"scanned {total_blocks:,} blocks of {block} bytes across {len(files)} files")
    print(f"  blocks shared by >=2 files : {len(cross)}")

    if cross:
        print("\nVERDICT: cross-file collision found — shared key under ECB-like mode is essentially certain.")
        for b, idxs in list(cross.items())[:5]:
            names = sorted(files[i][0].path.name for i in idxs)
            print(f"  block {b.hex()} → {names}")
    else:
        print("\nno cross-file block collisions — inconclusive (mode is probably not ECB, or plaintext is compressed first).")

    return {"n_collisions": len(cross), "collisions": cross}

### Test 3 — XOR-of-bodies entropy

If EOS used a stream-cipher mode (CTR, OFB, GCM) and the same key **and** the same nonce were reused across two files, then

$$ C_1 \oplus C_2 \;=\; (P_1 \oplus K_{\text{stream}}) \oplus (P_2 \oplus K_{\text{stream}}) \;=\; P_1 \oplus P_2 $$

and the result will have the entropy of XOR'ed plaintexts — typically well below 8 bits/byte, often with readable cribs. We compute Shannon entropy of `C_i ⊕ C_j` for every pair and flag anything below 7.5 bits/byte.

In [7]:
def _entropy(b: bytes) -> float:
    if not b:
        return 0.0
    counts = collections.Counter(b)
    n = len(b)
    return -sum((v / n) * math.log2(v / n) for v in counts.values())


def test_xor_entropy(files: list[tuple[EosparHeader, bytes]], threshold: float = 7.5, sample: int = 200_000):
    if len(files) < 2:
        print("need at least 2 files for the XOR entropy test")
        return None

    print(f"baseline single-file entropies (over up to {sample:,} bytes):")
    for hdr, body in files:
        H = _entropy(body[:sample])
        print(f"  {hdr.path.name:50s}  {H:.4f} bits/byte")

    print("\npairwise C_i XOR C_j entropies:")
    flagged = []
    for i in range(len(files)):
        for j in range(i + 1, len(files)):
            a = files[i][1][:sample]
            b = files[j][1][:sample]
            n = min(len(a), len(b))
            xored = bytes(x ^ y for x, y in zip(a[:n], b[:n]))
            H = _entropy(xored)
            tag = "  <-- LOW (key+nonce reuse?)" if H < threshold else ""
            print(f"  {files[i][0].path.name}  XOR  {files[j][0].path.name}  =>  {H:.4f} bits/byte{tag}")
            if H < threshold:
                flagged.append((files[i][0].path.name, files[j][0].path.name, H))

    if flagged:
        print("\nVERDICT: at least one pair has sub-random XOR entropy — same key + same nonce reuse is likely.")
    else:
        print("\nno pair fell below threshold — rules out (key + nonce) reuse, but not shared key with different nonces.")
    return {"flagged_pairs": flagged}

### Test 4 — Layout fingerprint

Compare each file's record counts (LC / BC) and the distance from the start of the encrypted body to the next round size boundary. Files that share the encryption pipeline tend to share these structural quirks. This is *necessary* (not sufficient) for shared keying, and it surfaces files that were probably written by a different EOSPRINT version.

In [8]:
def test_layout_fingerprint(files: list[tuple[EosparHeader, bytes]]):
    print(f"{'file':50s}  {'LC':>3s}  {'BC':>3s}  {'hdr':>4s}  {'body':>10s}  {'body%16':>7s}")
    fingerprints = []
    for hdr, body in files:
        fp = (hdr.n_laser_configs, hdr.n_build_configs, len(hdr.header_bytes), len(body) % 16)
        fingerprints.append(fp)
        print(f"{hdr.path.name:50s}  {hdr.n_laser_configs:3d}  {hdr.n_build_configs:3d}  {len(hdr.header_bytes):4d}  {len(body):10,d}  {len(body) % 16:7d}")

    consistent = len(set(fingerprints)) == 1
    print()
    if consistent:
        print("layout fingerprints are identical — consistent with one encryption pipeline (necessary, not sufficient, for shared key).")
    else:
        # Are at least the (LC, BC, body%16) triplets consistent? If LC/BC differ that's just different content.
        modlens = {fp[3] for fp in fingerprints}
        if len(modlens) == 1:
            print("record counts vary (different content) but body-length-mod-16 is consistent — still consistent with a shared cipher mode.")
        else:
            print("layouts differ in ways that suggest different format versions — comparing keys across these is unreliable.")
    return {"fingerprints": fingerprints, "consistent": consistent}

### Combined verdict

`shared_key_verdict()` runs all four tests and produces a single judgement on a 5-point scale:

- **PROVEN_SHARED** — a cross-file ECB collision (test 2) was found, **or** a sub-random XOR entropy was found (test 3). Either is conclusive.
- **STRONG_EVIDENCE_SHARED** — test 1 found a long constant random-looking run in the headers (key-id / KCV style) and the layout fingerprints (test 4) are consistent.
- **CONSISTENT_WITH_SHARED** — layout is consistent and headers vary only in fields that look like per-file IVs/salts. Could be shared key, can't prove it.
- **EVIDENCE_AGAINST** — a key-id-shaped field exists at the same offset across files but with different values.
- **INCONCLUSIVE** — nothing diagnostic showed up (most likely if EOS did the right thing: AES-GCM with per-file random nonces and per-file wrapped keys).

In [9]:
def shared_key_verdict(paths: Iterable[str | Path]):
    files = load_eospar_files(paths)
    if len(files) < 2:
        print("need at least 2 .eospar files")
        return

    print("=" * 70)
    print("PER-FILE HEADER SUMMARY")
    print("=" * 70)
    for hdr, _ in files:
        print(hdr.summary())
        print()

    print("=" * 70); print("TEST 1 — header diff");          print("=" * 70)
    r1 = test_header_diff(files)
    print()
    print("=" * 70); print("TEST 2 — cross-file ECB blocks"); print("=" * 70)
    r2 = test_block_collisions(files)
    print()
    print("=" * 70); print("TEST 3 — XOR-of-bodies entropy"); print("=" * 70)
    r3 = test_xor_entropy(files)
    print()
    print("=" * 70); print("TEST 4 — layout fingerprint");    print("=" * 70)
    r4 = test_layout_fingerprint(files)
    print()

    # Roll up.
    if r2 and r2["n_collisions"] > 0:
        verdict = "PROVEN_SHARED (ECB collision across files)"
    elif r3 and r3["flagged_pairs"]:
        verdict = "PROVEN_SHARED (key+nonce reuse detected via XOR entropy)"
    elif r1 and r1["long_constant_runs"] and r4 and r4["consistent"]:
        verdict = "STRONG_EVIDENCE_SHARED (constant key-id-shaped header field + consistent layout)"
    elif r4 and r4["consistent"]:
        verdict = "CONSISTENT_WITH_SHARED (layout matches; no positive proof of shared key)"
    else:
        verdict = "INCONCLUSIVE"

    print("=" * 70)
    print(f"VERDICT: {verdict}")
    print("=" * 70)
    return verdict

### Running the verdict

Drop your `.eospar` files into a folder and point `shared_key_verdict()` at them. Two or more files are required; more files give the tests more leverage (especially test 1).

```python
paths = sorted(Path("path/to/your/eospar_collection").glob("*.eospar"))
shared_key_verdict(paths)
```

The cell below runs the verdict on a one-element list (the demo file shipped with the repo), purely as a smoke test of the plumbing — it will short-circuit and remind you that two files are required.

In [10]:
shared_key_verdict([DEMO_EOSPAR])

need at least 2 .eospar files


### How to read the output

- A **PROVEN_SHARED** verdict from test 2 or test 3 is unambiguous — those tests have essentially zero false-positive rate at this data scale.
- A **STRONG_EVIDENCE_SHARED** verdict means the headers expose a constant random-looking field of ≥8 bytes at the same offset in every file. That is the canonical "key check value" / "key id" pattern. It does not *prove* the data is encrypted under that key (it could be a key-encrypting-key id, with per-file file-keys), but in practice the operational answer is the same: knowing one key would let you decrypt all of them.
- A **CONSISTENT_WITH_SHARED** verdict is the most common outcome when a vendor has done their crypto correctly (AES-GCM with per-file random nonces, per-file file-keys wrapped under the master key). You can't tell from ciphertext alone, and the only way forward is whatever metadata the plaintext header chooses to expose.
- An **EVIDENCE_AGAINST** verdict (constant-position, varying-value, random-looking field) is a sign that EOS is using a per-file or per-license key.
- An **INCONCLUSIVE** verdict means the format is heterogeneous across the files you supplied — try grouping by EOSPRINT version (visible in the parent `.openjob`) before re-running.